In [3]:
#Criar e abri o navegador

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd


driver = webdriver.Edge()

In [ ]:
# Carregar base de dados

df_produtos = pd.read_excel("Produtos.xlsx") 
df_produtos["link"] = df_produtos["link"].astype(str)
df_produtos["preco"] = df_produtos["preco"].astype(float)


# Função para pesquisar no gutenberg
def persquisar_gutenberg(nome, autor, driver):
    lista_palavras_autor = autor.split(" ")

    driver.get("https://www.gutenberg.org/")

    driver.find_element("class name", "search-input").send_keys(nome)
    driver.find_element("class name", "search-input").send_keys(Keys.ENTER)


    # selecionar os itens:

    lista_resultados = driver.find_elements("class name", "booklink")
    link = None
    for resultado in lista_resultados:
        texto = resultado.text
        if nome.lower() in texto.lower():
            if all(palavra in texto for palavra in lista_palavras_autor):
                link = resultado.find_element("class name", "link").get_attribute("href")
                break
    if link:
        preco = 0
    else:
        preco = None
    return link, preco


# Função para pesquisar no bookstoscrape
def pesquisar_bookstoscrape(nome, autor, categoria, driver):
    driver.get("https://books.toscrape.com/")

  
    # Aqui caso o site demore a carregar alguma vez, não vai quebrar o programa, pois tem um tempo de delay 
    lista_categorias = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(("class name", "nav-list"))
    )

    try:
        lista_categorias.find_element("link text", categoria).click()
    except:
        print("Nenhuma categoria selecionada")

    
    link = None
    preco = None
    encontrado = False


    # Aqui percorrer todas as páginas ate encontrar, caso não encontre o fluxo segue normal
    while not encontrado:

        lista_resultados = driver.find_elements("class name", "product_pod")

        for resultado in lista_resultados:
            elemento = resultado.find_element("tag name", "h3").find_element("tag name", "a")
            titulo = elemento.get_attribute("title")
            if nome.lower() in titulo.lower():
                encontrado = True
                link = elemento.get_attribute("href")
                preco = resultado.find_element("class name", "price_color").text
                break
        try:
            driver.find_element("link text", "next").click()
        except:
            break
    return link, preco

# Aqui filtro a busca através da base de dados
for linha in df_produtos.index:
    nome = df_produtos.loc[linha, "nome"]
    autor = df_produtos.loc[linha, "autor"]
    categoria = df_produtos.loc[linha, "categoria"]

    link1, preco1 = persquisar_gutenberg(nome, autor, driver)
    print(nome, link1, preco1)

    if not link1:

        link2, preco2 = pesquisar_bookstoscrape(nome, autor, categoria, driver)
        print(nome, link2, preco2)

        if preco2:
            preco2 = preco2.replace("£", "").replace(",", ".")
            preco2 = float(preco2)


        df_produtos.loc[linha, "preco"] = preco2
        df_produtos.loc[linha, "link"] = link2
    else:
        df_produtos.loc[linha, "preco"] = preco1
        df_produtos.loc[linha, "link"] = link1
    


Frankenstein https://www.gutenberg.org/ebooks/84 0
Romeo and Juliet https://www.gutenberg.org/ebooks/1513 0
The Great Gatsby https://www.gutenberg.org/ebooks/64317 0
Algorithms to Live By None None
Algorithms to Live By https://books.toscrape.com/catalogue/algorithms-to-live-by-the-computer-science-of-human-decisions_880/index.html £30.81
Sapiens None None
Sapiens https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html £54.23
Smarter Faster Better None None
Smarter Faster Better https://books.toscrape.com/catalogue/smarter-faster-better-the-secrets-of-being-productive-in-life-and-business_543/index.html £38.89


In [ ]:
# Crio uma nova tabela com os preços e links dos livros atualizado
df_produtos.to_excel("ProdutosAtualizado.xlsx", index = False)

